In [ ]:
# 1. Установка и импорт (выполняется один раз в среде)
# !pip install -q amplpy
# !python -m amplpy.modules install highs # Установка решателя HiGHS
from amplpy import AMPL

# 2. Исходные данные (те же, что и ранее)
tasks = {
    'b1': (15, 3, 9), 'b2': (14, 2, 5), 'b3': (11, 4, 7),
    'b4': (16, 6, 3), 'b5': (9, 2, 5), 'b6': (7, 3, 10),
    'b7': (8, 1, 9), 'b8': (9, 1, 7), 'b9': (12, 4, 6),
    'b10': (11, 3, 11), 'b11': (13, 7, 14), 'b12': (16, 8, 8)
}
# Зависимости (u -> v) и фиктивные дуги для корректной структуры
# (В AMPL мы можем задать их напрямую как множества)
edges = [
    (0, 1, 'b1'), (0, 2, 'b2'), (0, 3, 'b3'),
    (1, 4, 'b4'), (2, 5, 'b5'), (2, 6, 'b6'),
    (3, 7, 'd1'), (6, 7, 'd2'), (7, 8, 'b7'),
    (8, 9, 'b8'), (8, 10, 'b9'),
    (4, 11, 'd3'), (5, 11, 'd4'), (9, 11, 'd5'), (11, 12, 'b10'),
    (10, 13, 'd6'), (12, 13, 'd7'), (13, 14, 'b11'),
    (3, 15, 'd8'), (5, 15, 'd9'), (15, 16, 'b12'),
    (14, 17, 'd10'), (16, 17, 'd11')
]
S = 14 # Стоимость дня проекта

# 3. Инициализация AMPL
ampl = AMPL()
# Устанавливаем решатель HiGHS (бесплатный, входит в модули amplpy)
# В Google Colab можно использовать ampl_notebook(modules=["highs"])
ampl.solve(solver="highs") # По умолчанию будет использован встроенный решатель

# 4. Декларативное описание модели на AMPL
ampl.eval(r"""
    # Множества
    set NODES;                 # Узлы сети
    set EDGES within {NODES, NODES}; # Дуги (i,j)
    set TASKS;                 # Названия работ (включая фиктивные)

    # Параметры
    param t_max {TASKS};       # Максимальная продолжительность (t_пес)
    param t_min {TASKS};       # Минимальная продолжительность (t_опт)
    param cost {TASKS};        # Стоимость сокращения на 1 день
    param S;                   # Стоимость одного дня проекта
    param T_target;            # Целевое время выполнения

    # Переменные
    var x {TASKS} >= 0;        # Величина сокращения работы
    var y {NODES} >= 0;        # Время наступления события

    # Целевая функция: минимизировать стоимость сокращений + стоимость дней проекта
    minimize Total_Cost:
        sum {k in TASKS} cost[k] * x[k] + S * y[last(NODES)];

    # Ограничения
    # 1. Время события v >= время события u + продолжительность работы (i,j)
    subject to Precedence {(u,v,k) in EDGES}:
        y[v] >= y[u] + (t_max[k] - x[k]);

    # 2. Целевое время выполнения проекта (минимально возможное)
    subject to Target_Time:
        y[last(NODES)] <= T_target;

    # 3. Время сокращения не может превышать разницу между t_max и t_min
    subject to Max_Crash {k in TASKS}:
        x[k] <= t_max[k] - t_min[k];
""")

# 5. Загрузка данных в AMPL (используя Python-словари)
# Преобразуем данные в удобный для AMPL формат
node_set = sorted(list(set([u for u,v,k in edges] + [v for u,v,k in edges])))
task_data = [(k, t_max, t_min, cost) for k, (t_max, t_min, cost) in tasks.items()]
# Добавляем фиктивные работы
for u, v, k in edges:
    if k.startswith('d'):
        task_data.append((k, 0, 0, 0))

# Устанавливаем параметры через amplpy
ampl.set["NODES"] = node_set
ampl.set["TASKS"] = [t[0] for t in task_data]
ampl.set["EDGES"] = [(u, v, k) for u, v, k in edges]

# Загружаем параметры
for name, t_max, t_min, cost in task_data:
    ampl.param["t_max"][name] = t_max
    ampl.param["t_min"][name] = t_min
    ampl.param["cost"][name] = cost

# Вычисляем минимально возможное время (при всех t_min) с помощью AMPL
# Создадим временную модель для расчета
ampl.eval("""
    var y_min {NODES} >= 0;
    minimize Dummy: 0;
    subject to Prec_Min {(u,v,k) in EDGES}:
        y_min[v] >= y_min[u] + t_min[k];
""")
ampl.solve()
T_min_possible = ampl.get_value("y_min[last(NODES)]")

# Устанавливаем целевое время и стоимость дня
ampl.param["T_target"] = T_min_possible
ampl.param["S"] = S

# 6. Решение основной задачи
ampl.solve()

# 7. Извлечение и вывод результатов
if ampl.solve_result == "solved":
    print("="*60)
    print("ЗАДАЧА 3: ОПТИМИЗАЦИЯ СТОИМОСТИ (решение через AMPL)")
    print("="*60)
    print(f"Минимально возможный срок (при всех t_опт) = {T_min_possible:.0f} дней.")
    print(f"Минимальная стоимость проекта при этом сроке = {ampl.get_value('Total_Cost'):.2f} ден. ед.\n")
    
    print("Оптимальные продолжительности работ:")
    # Получаем значения переменных x (сокращения)
    x_values = ampl.get_data("x").to_pandas()
    for task in tasks.keys():
        reduction = x_values.loc[task, 'x']
        duration = tasks[task][0] - reduction
        print(f"  {task}: {duration:.2f} дней (сокращено на {reduction:.2f})")
else:
    print("Решение не найдено.")

No variables declared.


[WARNING] 
	line 20 offset 1133
	last may only be used with ordered or circular sets
	context:  sum {k in TASKS} cost[k] * x[k] + S *  >>> y[last(NODES) <<< ];
[WARNING] 
	line 24 offset 1341
	3 indices for set of dimen 2
	context:  subject to Precedence {(u,v,k) in  >>> EDGES} <<< :
[WARNING] 
	line 29 offset 1541
	last may only be used with ordered or circular sets
	context:   >>> y[last(NODES) <<< ] <= T_target;


AMPLException: line 20 offset 1133
last may only be used with ordered or circular sets
context:  sum {k in TASKS} cost[k] * x[k] + S *  >>> y[last(NODES) <<< ];